# Structured Data - Lung Cancer and WHO Data Ingestion

In [0]:
%pip install structlog
dbutils.library.restartPython()

In [0]:
import structlog, logging, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

logging.basicConfig(level=logging.INFO)
log = structlog.get_logger()

def log_ingest(table_name, record_count, source_path, status, error=None):
    entry = {
        'timestamp': datetime.datetime.utcnow().isoformat(),
        'pipeline_stage': 'bronze',
        'table': table_name,
        'records': record_count,
        'source': source_path,
        'status': status,
        'error': str(error) if error else None
    }
    log.info('ingest_event', **entry)
    spark.createDataFrame([entry]).write.format('delta').mode('append').saveAsTable('bronze.pipeline_logs')

print("Logging ready")

In [0]:
display(dbutils.fs.ls('/Volumes/workspace/bronze/raw_files'))

In [0]:
start = datetime.datetime.utcnow()
source = '/Volumes/workspace/bronze/raw_files/lung_cancer_dataset.csv'

df_lung = spark.read.csv(source, header=True, inferSchema=True)

df_lung = df_lung \
    .withColumn('_ingest_timestamp', F.current_timestamp()) \
    .withColumn('_source_file', F.lit(source))

df_lung.write.format('delta').mode('overwrite').saveAsTable('bronze.raw_lung_cancer')

print(f'Done! {df_lung.count():,} rows saved')

In [0]:
df = spark.read.csv('/Volumes/workspace/bronze/raw_files/lung_cancer_dataset.csv', header=True)
print(df.columns)
print("\nRows:", df.count())

In [0]:
source_who = '/Volumes/workspace/bronze/raw_files/WHO_pollution.csv'

df_who = spark.read.csv(source_who, header=True, inferSchema=True, sep=';')
df_who = df_who.withColumn('_ingest_timestamp', F.current_timestamp()) \
               .withColumn('_source_file', F.lit(source_who))

df_who.write.format('delta').mode('overwrite').saveAsTable('bronze.raw_who_pollution')

print(f'Done! {df_who.count():,} rows saved')

In [0]:
source_pubmed = '/Volumes/workspace/bronze/raw_files/abstract-lungcancer.txt'

df_pubmed = spark.read.text(source_pubmed)
df_pubmed = df_pubmed.withColumn('_ingest_timestamp', F.current_timestamp()) \
                     .withColumn('_source_file', F.lit(source_pubmed))

df_pubmed.write.format('delta').mode('overwrite').saveAsTable('bronze.raw_pubmed')

print(f'Done! {df_pubmed.count():,} lines saved')

In [0]:
tables = ['bronze.raw_lung_cancer', 'bronze.raw_who_pollution', 'bronze.raw_pubmed']

for table in tables:
    count = spark.table(table).count()
    print(f'{table}: {count:,} rows')

## Unstructured Data - PubMed Abstracts

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType

raw = spark.read.text('/Volumes/workspace/bronze/raw_files/abstract-lungcancer.txt')
print(f"Total lines: {raw.count():,}")

In [0]:
ext = '\n'.join([row.value for row in raw.collect()])

def split_abstracts_by_sequential_numbers(text):

    pattern = re.compile(r'(?m)^(\d+)\. ')
    matches = list(pattern.finditer(text))
    
    if not matches:
        return []

    sequential_positions = []
    expected_num = 1

    for match in matches:
        num = int(match.group(1))
        if num == expected_num:
            sequential_positions.append(match.start())
            expected_num += 1

    abstracts = []
    for i, start in enumerate(sequential_positions):
        end = sequential_positions[i + 1] if i + 1 < len(sequential_positions) else len(text)
        abstracts.append(text[start:end].strip())

    return abstracts

abstracts = split_abstracts_by_sequential_numbers(ext)

records = []
for ab in abstracts:
    if not ab:
        continue
    lines = ab.split('\n')
    records.append({
        'raw_text': ab,
        'first_line': lines[0] if lines else None
    })

print(f"Total abstracts: {len(records):,}")

In [0]:
df_pubmed = spark.createDataFrame(records)
df_pubmed.write.format('delta').mode('overwrite').saveAsTable('bronze.raw_pubmed_abstracts')
print(f"Saved {df_pubmed.count():,} abstracts")

In [0]:
RISK_TERMS = [
    'smoking', 'tobacco', 'cigarette', 'radon', 'asbestos',
    'pm2.5', 'pm10', 'particulate', 'pollution', 'carcinogen',
    'exposure', 'occupational', 'genetic', 'mutation',
    'adenocarcinoma', 'squamous', 'small cell', 'non-small cell'
]

def extract_keywords(text):
    if not text:
        return []
    return [term for term in RISK_TERMS if term in text.lower()]

extract_udf = F.udf(extract_keywords, ArrayType(StringType()))

df_keywords = df_pubmed \
    .withColumn('risk_keywords', extract_udf(F.col('raw_text'))) \
    .withColumn('keyword_count', F.size(F.col('risk_keywords')))

df_keywords.write.format('delta').mode('overwrite').saveAsTable('bronze.raw_pubmed_keywords')
print(f"Saved {df_keywords.count():,} records with keywords")
display(df_keywords.select('first_line', 'risk_keywords', 'keyword_count').limit(5))

## Streaming Data - Air Quality API

In [0]:
import requests, datetime
from pyspark.sql import functions as F

COUNTRIES = [
    {'country': 'United States', 'lat': 40.71, 'lon': -74.01},
    {'country': 'China',         'lat': 39.90, 'lon': 116.40},
    {'country': 'India',         'lat': 28.61, 'lon': 77.21},
    {'country': 'Brazil',        'lat': -23.55,'lon': -46.63},
    {'country': 'Germany',       'lat': 52.52, 'lon': 13.40},
    {'country': 'United Kingdom','lat': 51.51, 'lon': -0.13},
    {'country': 'Japan',         'lat': 35.69, 'lon': 139.69},
    {'country': 'Australia',     'lat': -33.87,'lon': 151.21},
    {'country': 'Russia',        'lat': 55.75, 'lon': 37.62},
    {'country': 'South Africa',  'lat': -26.20,'lon': 28.04},
    {'country': 'Nigeria',       'lat': 6.52,  'lon': 3.38},
    {'country': 'Mexico',        'lat': 19.43, 'lon': -99.13},
    {'country': 'Argentina',     'lat': -34.60,'lon': -58.38},
    {'country': 'Finland',       'lat': 60.17, 'lon': 24.94},
]

def fetch_air_quality(country):
    try:
        r = requests.get(
            'https://air-quality-api.open-meteo.com/v1/air-quality',
            params={
                'latitude':  country['lat'],
                'longitude': country['lon'],
                'hourly':    'pm10,pm2_5',
                'past_days': 4,
                'forecast_days': 1,
            }, timeout=10
        )
        data = r.json()['hourly']
        return [
            {
                'country':     country['country'],
                'time':        data['time'][i],
                'pm2_5':       data['pm2_5'][i],
                'pm10':        data['pm10'][i],
                '_fetched_at': datetime.datetime.now(datetime.UTC).isoformat()
            }
            for i in range(len(data['time']))
        ]
    except Exception as e:
        print(f"Failed {country['country']}: {e}")
        return []

print("Ready")

In [0]:
all_records = []
for c in COUNTRIES:
    records = fetch_air_quality(c)
    all_records.extend(records)
    print(f"{c['country']}: {len(records)} records")

print(f"\nTotal: {len(all_records):,} records fetched")

df_aq = spark.createDataFrame(all_records)
df_aq.write.format('delta').mode('overwrite').saveAsTable('bronze.raw_air_quality')

print("Saved to bronze.raw_air_quality")